# Urdu OCR Project | Code Saviours SI-26 | [Your Full Name]

**Week 1 — Set up tools, understand the project, collect data.**

This notebook follows the Week 1 handout (Part A/B/C) but uses this repo's
`src/synthetic.py` to *bootstrap* your 100-image minimum in seconds, so you
can spend your time on real photographed images (which the handout itself
weights most heavily for robustness) instead of manually generating 20
synthetic ones by hand.


## Part A — Accounts & tools

Do Steps 1-4 from the handout (GitHub repo, HuggingFace account, Colab, connect Colab to GitHub) outside this notebook, then come back here.

In [ ]:
# Run once per Colab session (safe to re-run: pulls latest instead of
# silently reusing a stale checkout from an earlier run this session).
import os
import subprocess
import sys

REPO_DIR = '/content/repo'
# IMPORTANT: pinned to this specific branch, not the repo's default branch
# -- see Quickstart_Colab.ipynb's Cell 1 for why (an unqualified clone/pull
# can silently track a branch that's missing later fixes, and `git pull`
# then reports "Already up to date" against the wrong branch with no
# warning). Replace with your own fork's URL/branch if running elsewhere,
# or with 'main' once this branch has actually been merged there.
BRANCH = 'claude/high-accuracy-solution-r9r2t1'
if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} --single-branch https://github.com/Rameesha-Malik/ML-SI-Solution-Urdu-OCR.git {REPO_DIR}
else:
    print(f'{REPO_DIR} already exists -- fetching + resetting to origin/{BRANCH} instead of re-cloning.')
    !git -C {REPO_DIR} fetch origin {BRANCH}
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} reset --hard origin/{BRANCH}

os.chdir(REPO_DIR)
!git -C {REPO_DIR} log -1 --format='Repo HEAD: %h %s'

# Deliberately NOT --upgrade-strategy eager, and NOT --force-reinstall on
# the whole file: both would make pip consider reinstalling `torch` even
# though Colab's pre-installed torch already satisfies our ">=2.2.0" bound
# -- and a freshly pip-resolved torch build was observed, in testing, to be
# missing a CUDA sub-package it needs (ImportError: libcublas.so.13), which
# would trade a fixable Python-level bug for a broken GPU runtime. Plain
# `--upgrade -r requirements.txt` still fully resolves and changes
# transformers/pillow/gradio (their pins are strict enough that pip must
# change them regardless of strategy -- upgrade-strategy only affects
# whether *unnamed transitive* dependencies like torch get proactively
# touched), while leaving torch itself alone.
!pip install -q --upgrade -r requirements.txt

# Ground-truth check of what's on disk, via a subprocess (`pip show`) --
# NOT `import transformers`/`import PIL`. If a notebook cell earlier in
# this session already imported one of these, Python caches that module
# in memory for the rest of the process; reinstalling on disk does not
# change what's already loaded there. The actual training/eval scripts
# run as a separate `!python ...` subprocess each time, so they always
# see what's genuinely on disk -- this mirrors that rather than this
# kernel's own (possibly stale) view.
def _installed_version(pkg):
    result = subprocess.run(['pip', 'show', pkg], capture_output=True, text=True)
    return next(
        (l.split(':', 1)[1].strip() for l in result.stdout.splitlines() if l.startswith('Version:')),
        '?',
    )

versions = {pkg: _installed_version(pkg) for pkg in ('transformers', 'pillow', 'gradio')}
print('Installed on disk:', versions)

if 'src' not in sys.path:
    sys.path.insert(0, 'src')

expected_major = {'transformers': ('4.',), 'pillow': ('11.',), 'gradio': ('4.', '5.', '6.0', '6.1')}
bad = {
    p: v for p, v in versions.items()
    if not any(v.startswith(prefix) for prefix in expected_major[p])
}
if bad:
    raise RuntimeError(
        f'Unexpected version(s) on disk: {bad} (expected transformers 4.x, '
        'pillow 11.x, gradio <6.20 per requirements.txt). pip could not '
        'fully satisfy the pins here -- likely another package in this '
        'Colab image forces a conflicting version. Re-run this cell and '
        'share the FULL pip output above (not just this message) so the '
        'real conflicting package can be identified.'
    )
print('Setup complete -- all pinned versions correctly installed on disk.')


## Part B — Research task

Answer in your own words (the notes below are a starting point — personalise them,
don't submit them verbatim).

**What is OCR (Optical Character Recognition)?**
OCR is the process of converting an image that contains text — a scanned page, a
photo of a signboard, a screenshot — into machine-readable, editable text. It
combines computer vision (finding where the text is and what shape each
character has) with language modelling (turning those shapes into the most
likely sequence of characters/words), so the output isn't just "this pixel
pattern looks like a letter" but "these pixel patterns, read together, spell
this sentence."

**Why is Urdu OCR harder than English OCR?**
Urdu is normally written in the Nastaliq calligraphic style, where letters
change shape depending on their position in a word and are frequently stacked
diagonally rather than sitting on one flat baseline the way Latin letters do —
so there is no single consistent "line" to segment. Many letters are
distinguished only by the number/placement of small dots, and several
different Unicode code points can render as visually identical letters,
which confuses both segmentation and text-matching. On top of that, large,
clean, publicly available labelled Urdu OCR datasets are far rarer than
English ones, so most off-the-shelf OCR engines (Tesseract included, see
Week 2) were never trained on enough Urdu data to do well.

**What are 2 real-world situations where Urdu OCR would be useful?**
1. Digitising government, legal, and historical Urdu documents (land
   records, court judgments, newspapers) so they become searchable and
   preservable instead of sitting as untouched paper or scanned images.
2. Accessibility and automation for everyday use — e.g. automatically
   extracting and translating Urdu text from photographed signboards or
   restaurant menus for a traveller, or letting a visually impaired user's
   phone read out Urdu text from a photo via text-to-speech.


## Part C — Data collection

### Step 6a — Bootstrap with synthetic images (fast, automatic)

`src/synthetic.py` renders real, curated Urdu sentences (`data/corpus/urdu_sentences.txt`) in 3 different Urdu fonts (Nastaliq / Naskh / Sans — the "different fonts" variety the handout asks for), varied backgrounds, sizes, rotation, noise and JPEG artifacts, so a single command gives you a large, varied starting dataset.

In [ ]:
from synthetic import generate_dataset

stats = generate_dataset(
    output_dir='data/raw/synthetic',
    labels_csv='data/labels_synthetic.csv',
    num_images=150,     # well over the Week 1 minimum of 100
    pseudo_fraction=0.3,
    seed=42,
)
print(f'Generated {stats.total} images ({stats.from_corpus} corpus + {stats.from_pseudo} pseudo)')
print('Fonts used:', stats.fonts_used)


### Step 6b — Add real photographed / downloaded images

Synthetic data alone will not make an accurate model — real photos, screenshots, and downloaded-dataset images teach the model what *actually messy* input looks like. Follow the handout's Source 1 (existing datasets) and Source 2 (photograph newspapers/books/signboards, screenshot Urdu news sites) and drop the files into the matching `data/raw/<category>/` folder created below.

In [ ]:
import os
folders = [
    'data/raw/newspaper', 'data/raw/books', 'data/raw/signboards',
    'data/raw/synthetic', 'data/raw/other',
]
for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f'Created: {folder}')
print('Folder structure ready! Upload your real images into the matching folders above.')


### Step 7 — Label your real images

For every real image you added, add one row to `data/labels_real.csv` (`image,text` — same two columns as the handout's `data/labels.csv`).

In [ ]:
import csv

# Fill in your actual filenames + the Urdu text you see in each image.
labels = [
    # {'image': 'data/raw/newspaper/img001.jpg', 'text': '...'},
]

os.makedirs('data', exist_ok=True)
with open('data/labels_real.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['image', 'text'])
    writer.writeheader()
    writer.writerows(labels)
print(f'Real-image labels file created with {len(labels)} entries')


### Merge synthetic + real labels into one `data/labels.csv`

This is the single file every later notebook reads from.

In [ ]:
from synthetic import merge_labels_csv

n = merge_labels_csv(
    ['data/labels_synthetic.csv', 'data/labels_real.csv'],
    'data/labels.csv',
)
print(f'data/labels.csv ready with {n} total labelled images')


**Checklist before moving to Week 2:** `data/labels.csv` has 100+ rows, images actually exist at the listed paths, and you've committed your Week 1 notebook + `data/` to GitHub.